<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/hybrid-rag-poc/validacao_estrutura_chunks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Instalar e Importar
!pip install -q chromadb
import chromadb
from google.colab import drive

# 2. Montar o Drive
drive.mount('/content/drive')

# 3. Aponte para a pasta do banco
DB_PATH = "/content/drive/MyDrive/pln/hybrid-rag-poc/data/chroma_db_jo"

# 4. Inicialize o cliente
client = chromadb.PersistentClient(path=DB_PATH)

# 5. Listar as coleções
collections = client.list_collections()

if len(collections) > 0:
    print("Coleções encontradas no banco:")
    for collection in collections:
        print(f"- {collection.name}")
else:
    print("Nenhuma coleção foi encontrada nesta pasta. Verifique se o caminho do DB_PATH está correto.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Coleções encontradas no banco:
- comentario_moody_jo


In [5]:
import chromadb
import pandas as pd
from google.colab import drive

# 2. Configurações
COLLECTION_NAME = "comentario_moody_jo" # Ajuste aqui

# 3. Carregar o Banco
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name=COLLECTION_NAME)

# 4. Extrair todos os dados
# O método get() sem argumentos traz tudo
data = collection.get(include=["documents", "metadatas"])

# 5. Criar DataFrame
df = pd.DataFrame({
    'id': data['ids'],
    'document': data['documents'],
    'metadatas': data['metadatas']
})

# 6. Análise de Duplicidade (O ponto principal da sua dúvida)
# Isso mostrará linhas onde o conteúdo do 'document' é exatamente o mesmo
duplicates = df[df.duplicated(subset=['document'], keep=False)].sort_values(by='document')

print(f"Total de chunks carregados: {len(df)}")
print(f"Total de chunks duplicados encontrados: {len(duplicates)}")

# 7. Visualização
if not duplicates.empty:
    print("\n--- Amostra de Chunks Duplicados ---")
    # Exibe os IDs e metadados associados a chunks idênticos
    print(duplicates[['id', 'document', 'metadatas']].head(10))
else:
    print("\nNenhum conteúdo (document) duplicado foi encontrado.")

# 8. Opcional: Salvar em Excel para análise externa
# duplicates.to_excel("chunks_duplicados.xlsx", index=False)
# print("\nRelatório de duplicatas salvo como 'chunks_duplicados.xlsx'")

Total de chunks carregados: 275
Total de chunks duplicados encontrados: 0

Nenhum conteúdo (document) duplicado foi encontrado.


In [6]:
import pandas as pd

# 1. Transformar a coluna 'metadatas' em colunas separadas
# Isso supõe que os metadados sejam dicionários (ex: {'livro': 'Genesis', 'verso': '1:1'})
meta_df = pd.json_normalize(df['metadatas'])

# 2. Unir com os dados principais
df_final = pd.concat([df[['id', 'document']], meta_df], axis=1)

# 3. Exibir uma prévia das primeiras linhas para você verificar as colunas
print("Estrutura dos dados (Ampliados):")
print(df_final.head(10))

# 4. Salvar em Excel
output_file = "/content/drive/MyDrive/pln/hybrid-rag-poc/chunks/auditoria_chunks.xlsx"
df_final.to_excel(output_file, index=False)

print(f"\nArquivo salvo com sucesso em: {output_file}")
print("Dica: Baixe esse arquivo e abra no Excel para filtrar as colunas de metadados.")

Estrutura dos dados (Ampliados):
           id                                           document nivel_3  \
0  chunk_0000                                         INTRODUÇÃO           
1  chunk_0001  Título \nO nome do livro e do seu herói, 'iyyô...           
2  chunk_0002  Gênero Literário\nA essência do livro é poesia...           
3  chunk_0003  Autoria e Data\nAs discussões sobre a autoria ...           
4  chunk_0004  Tema\nAtravés do problema da teodicéia, o livr...           
5  chunk_0005                                             ESBOÇO           
6  chunk_0006  I. Desolação : A provação da sabedoria de Jó. ...           
7  chunk_0007  A. Descrição da sabedoria de Jó. 1:1-5.\n\tB. ...           
8  chunk_0008                                         COMENTÁRIO           
9  chunk_0009  I. Desolação: A Provação da Sabedoria de Jó. 1...           

  referencia_versiculo              tipo nivel_2  \
0                       comentario_geral           
1                       co